In [ ]:
import time
import math
import random
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

import torch
from scipy.stats import norm
import gpytorch
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.svm import SVR
from scipy.stats import qmc  # for Latin hypercube sampling
from pyDOE import lhs
import time
import matplotlib as mpl
from scipy.stats.qmc import Sobol
from kan import *
# from kan1 import KAN

# ---- import your WO model class (must be on PYTHONPATH) ----
from Gpmodel import BaseGP
from NNGPmodel import NNGP
from KANGPmodel import KANGP
import os
import win32com.client as win32

In [ ]:
import pandas as pd
# 1️⃣ 读取 Excel 文件
file_path = "ORC_LHS_Simulation_inistial_data100.xlsx"  # 你的文件名
df = pd.read_excel(file_path)
# 2️⃣ 选择前三列和最后两列
df = df.iloc[:, [0, 1, 2, -2, -1]]
# 3️⃣ 只保留 Converged == True 的数据
df = df[df["Converged"] == True]
# 4️⃣ 构造特征 X 和目标 y
X = df.iloc[:, 0:3].values  # 三个特征：x1_Flow, x2_Pc, x3_Pp
y = df.iloc[:, 3].values    # 目标值：net_rev_USDyr
print("✅ 数据读取完成")
print("特征 X 形状:", X.shape)
print("目标 y 形状:", y.shape)
print("示例数据：")
print(df.head())
#数据集预处理和划分
#数据归一化
# 对 X 做归一化
scaler_X = MinMaxScaler().fit(X)
X_init_scaled = scaler_X.transform(X)
# 对 y 做归一化
scaler_y = MinMaxScaler().fit(y.reshape(-1, 1))
y_init_scaled = scaler_y.transform(y.reshape(-1, 1)).flatten()
x_initial = np.array(X_init_scaled, dtype=np.float32)
y_initial = np.array(y_init_scaled, dtype=np.float32)

# x_train, x_test, y_train, y_test = train_test_split(
#     x_initial, y_initial,
#     test_size=0.2,
#     random_state=47,      # 固定种子，结果可复现
#     shuffle=True
# )
def print_stats(name, data):
    """打印均值与标准差，保留 4 位小数"""
    mean = np.mean(data, axis=0)
    std  = np.std(data, axis=0)
    print(f"======== {name} ========")
    print("mean:", np.round(mean, 4))
    print("std :", np.round(std, 4))
    print()

# 一键打印 8 个量
print_stats("x_initial", x_initial)
print(x_initial.shape)
print_stats("y_initial", y_initial)
print(y_initial.shape)

x_initial = torch.tensor(x_initial, dtype=torch.float32)
y_initial  = torch.tensor(y_initial, dtype=torch.float32)


In [ ]:
#候选集 X的数值的候选集
# 二、变量与范围  Sobol采样

from scipy.stats.qmc import Sobol

param_bounds = {
    'x1_Flow': (140, 340),
    'x2_Pc':   (0.4, 1.5),
    'x3_Pp':   (4.0, 7.1),
}
n_samples = 100000

sampler = Sobol(d=len(param_bounds), scramble=True, seed=99)
samples = sampler.random_base2(m=int(np.ceil(np.log2(n_samples))))  # 用 2^m 个点
samples = samples[:n_samples]  # 截取前 n_samples 个

keys = list(param_bounds.keys())
X_candidate = np.zeros_like(samples)
for i, key in enumerate(keys):
    low, high = param_bounds[key]
    X_candidate[:, i] = low + samples[:, i] * (high - low)

X_candidate = np.round(X_candidate, 3)

print(X_candidate.shape)

#基于训练集的标准化，对候选集进行标准化
X_candidate_normalized = scaler_X.transform(X_candidate)
print("\nX_candidate_normalized  的均值 (每列):", np.mean(X_candidate_normalized, axis=0))
print("X_scaled 的标准差 (每列):", np.std(X_candidate_normalized, axis=0))
X_candidate_normalized = torch.tensor(X_candidate_normalized, dtype=torch.float32)

In [ ]:
import numpy as np
import torch

# 1. 定义你要加入的那个特定点 (确保顺序与 keys 一致：Flow, Pc, Pp)
new_point = np.array([[276.11, 0.403, 4.92]])

# 2. 将新点拼接到原始的 X_candidate 中
# 使用 np.vstack 将其添加到数组的末尾
X_candidate = np.vstack([X_candidate, new_point])

# 打印一下形状，你会发现样本数变成了 100001
print("加入新点后的 X_candidate 形状:", X_candidate.shape)

# 3. 基于训练集的标准化（Scaler），对包含新点的候选集进行统一标准化
# 这一步非常重要：必须先转换再变 Tensor，确保新点也符合同样的分布
X_candidate_normalized = scaler_X.transform(X_candidate)

# 4. 打印校验
print("\nX_candidate_normalized 的均值 (每列):", np.mean(X_candidate_normalized, axis=0))
print("X_candidate_normalized 的标准差 (每列):", np.std(X_candidate_normalized, axis=0))

# 5. 转换为 Torch Tensor
X_candidate_normalized = torch.tensor(X_candidate_normalized, dtype=torch.float32)

# 如果你想确认新点是否在最后一行：
print("\n新加入的点标准化后的数值:\n", X_candidate_normalized[-1])

In [ ]:
#模拟运行的相关函数
#对数温度
def calc_LMTD(Th_in, Th_out, Tc_in, Tc_out):
    """
    计算对数平均温差 (LMTD)
    输入单位: °C 或 K (只要一致)
    """
    dT1 = Th_in - Tc_out
    dT2 = Th_out - Tc_in
    if dT1 == dT2:
        return dT1
    else:
        return (dT1 - dT2) / np.log(dT1 / dT2)


#---展平
def flatten_bo_results(results_all, flags_all=None, verbose=False):
    """
    稳健版本：自动遍历所有 (r, iteration, batch) 实际存在的数据，
    不假设三维结构是规整方阵，因此不会越界。
    """

    rows = []

    # 确保是 list-of-lists-of-lists，不要强制 np.array
    #（np.asarray 会把列表变成 object array，有时破坏嵌套结构）
    def to_list(x):
        if isinstance(x, np.ndarray):
            return x.tolist()
        return x

    results_all = to_list(results_all)
    flags_all = to_list(flags_all) if flags_all is not None else None

    R = len(results_all)

    for r in range(R):
        if results_all[r] is None:
            continue

        iterations = results_all[r]
        I = len(iterations)

        for it in range(I):
            batch = iterations[it]
            if batch is None:
                continue

            B = len(batch)

            for j in range(B):

                # --- 读取 res ---
                try:
                    res = results_all[r][it][j]
                except Exception:
                    # 越界或 None
                    res = None

                # --- 读取 flag ---
                flag = False
                if flags_all is not None:
                    try:
                        flag = bool(flags_all[r][it][j])
                    except Exception:
                        flag = False

                # --- 基础记录 ---
                base = {
                    "run": r,
                    "iter": it,
                    "idx_in_batch": j,
                    "Converged": flag
                }

                # ==============================
                #   展平 res（与你原逻辑保持一致）
                # ==============================
                if isinstance(res, dict):

                    rec = base.copy()

                    # ----- inputs -----
                    if "inputs" in res and isinstance(res["inputs"], dict):
                        for k, v in res["inputs"].items():
                            rec[f"input_{k}"] = v

                    # ----- important_* -----
                    imp_keys = ["important_resluts", "important_results", "important_result", "important"]
                    found = False
                    for ik in imp_keys:
                        if ik in res and isinstance(res[ik], dict):
                            for k, v in res[ik].items():
                                rec[f"imp_{k}"] = v
                            found = True
                            break

                    # ----- 若没有 important，则平铺其它字段 -----
                    if not found:
                        for k, v in res.items():
                            if k == "inputs":
                                continue
                            if isinstance(v, dict):
                                for kk, vv in v.items():
                                    rec[f"{k}_{kk}"] = vv
                            else:
                                rec[k] = v

                    rows.append(rec)

                elif res is None:
                    # 失效点，保持占位
                    rows.append(base.copy())

                else:
                    # numpy / list / number
                    rec = base.copy()
                    if isinstance(res, (list, tuple, np.ndarray)):
                        arr = np.asarray(res).ravel()
                        for kk, vv in enumerate(arr):
                            rec[f"val_{kk}"] = vv
                    else:
                        rec["value"] = res
                    rows.append(rec)

                if verbose:
                    print(f"[OK] processed run={r}, it={it}, j={j}")

    # -----------------------
    # 生成 DataFrame
    # -----------------------
    df = pd.DataFrame(rows)

    # 优先列顺序
    priority = ["run", "iter", "idx_in_batch", "Converged"]
    rest = [c for c in df.columns if c not in priority]
    df = df[priority + rest]

    # 尝试把列转为数值
    for c in rest:
        df[c] = pd.to_numeric(df[c], errors='ignore')

    return df


#惩罚因子  
def pressure_ratio_penalty(pressure_ratio, base_PR=8.0, penalty_strength=0.12):
    """
    指数惩罚因子模型
    pressure_ratio: 当前压力比
    base_PR: 基准压力比（经济最优区间）
    penalty_strength: 惩罚强度系数 (0.1-0.5)
    """
    # 指数惩罚：压力比越高，惩罚越大
    penalty = np.exp(penalty_strength * (pressure_ratio - base_PR))
    return penalty


def run_aspen_batch(next_x_batch, Application, address):
    """
    运行 Aspen 模拟批次。
    ----------
    next_x_batch : ndarray, shape (N, 3)
        每一行对应一个采样点 (x1_Flow, x2_Pc, x3_Pp)
    Application : COM 对象
        已连接的 Aspen 对象
    address : str
        .bkp 文件所在目录（用于定位 .his 文件）

    返回:
        results_list : list[dict]
            每个收敛点的完整结果字典
        valid_flags : list[bool]
            是否收敛（True=收敛，False=错误/不收敛）
    """
    results_list = []
    valid_flags = []

    for i, (x1, x2, x3) in enumerate(next_x_batch):
        print(f"\n🔹 运行样本 {i+1}/{len(next_x_batch)}: x1={x1:.3f}, x2={x2:.3f}, x3={x3:.3f}")

        try:
            # 设置输入变量
            Application.Tree.FindNode(r"\Data\Streams\R1\Input\TOTFLOW\MIXED").Value = float(x1)
            Application.Tree.FindNode(r"\Data\Blocks\TURBINE\Input\PRES").Value = float(x2)
            Application.Tree.FindNode(r"\Data\Blocks\PUMP\Input\PRES").Value = float(x3)

            # 初始化并运行
            Application.Reinit()
            Application.Engine.Run2(1)

            # 等待运行结束
            while Application.Engine.IsRunning == 1:
                time.sleep(2)

            # ---- 检查是否收敛 ----
            runid = Application.Tree.FindNode("\Data\Results Summary\Run-Status\Output\RUNID").Value
            hisfile = address + '\\' + runid + '.his'
            with open(hisfile, 'r', encoding='gbk') as f:
                isError = isError = np.any(np.array([line.find('ERROR') for line in f.readlines()])>=0)
                valid_flags.append(not isError)

            # ---- 输出结果 ----
            T_out = Application.Tree.FindNode(r"\Data\Streams\R3\Output\TEMP_OUT\MIXED").Value
            Q_cond = abs(Application.Tree.FindNode(r"\Data\Blocks\CONGEN\Output\QCALC").Value)
            Q_evap = abs(Application.Tree.FindNode(r"\Data\Blocks\EVA\Output\HX_DUTY").Value)
            P_pump = abs(Application.Tree.FindNode(r"\Data\Blocks\PUMP\Output\ELEC_POWER").Value)
            P_turb = abs(Application.Tree.FindNode(r"\Data\Blocks\TURBINE\Output\BRAKE_POWER").Value)
            V_out = Application.Tree.FindNode(r"\Data\Streams\R3\Output\VOLFLMX\MIXED").Value
            T_hi = Application.Tree.FindNode(r"\Data\Streams\H1\Output\TEMP_OUT\MIXED").Value
            T_ho = Application.Tree.FindNode(r"\Data\Streams\H2\Output\TEMP_OUT\MIXED").Value
            T_ci = Application.Tree.FindNode(r"\Data\Streams\R1\Output\TEMP_OUT\MIXED").Value
            T_co = Application.Tree.FindNode(r"\Data\Streams\R2\Output\TEMP_OUT\MIXED").Value

            # ---- 工艺计算 ----
            Q_cond_kw = Q_cond * 4.184e-3
            Q_evap_kw = Q_evap * 4.184e-3
            Uc, Ue = 0.865, 0.568
            dT = calc_LMTD(T_hi, T_ho, T_ci, T_co)
            A_cond = Q_cond_kw / (Uc * dT)
            A_evap = Q_evap_kw / (Ue * dT)
            C_hex = 9367.8 * (A_cond ** 0.65 + A_evap ** 0.65)

            V_out_m3 = V_out * 1.6667 / 100000
            C_turb = 1.5 * (225 + 170 * V_out_m3)

            PCW = 0.03
            CP_WATER = 4.186
            DELTA_T_CW = 20.0
            HOURS_PER_YEAR = 8000
            m_dot_kg_s = Q_cond_kw / (CP_WATER * DELTA_T_CW)
            CCW = m_dot_kg_s * 3.6
            C_water = HOURS_PER_YEAR * PCW * CCW

            payback = 8
            C_equi = (C_hex + C_turb) / payback
            penalty = pressure_ratio_penalty(x3 / x2, base_PR=8.0, penalty_strength=0.12)
            C_equi = penalty * C_equi
            total_expend = C_equi + C_water

            PE = 0.1
            GC = P_turb - P_pump
            earnings = HOURS_PER_YEAR * PE * GC
            net_rev = earnings - total_expend

            if T_out <= 15:
                net_rev = 0.0

            result = {
                'x1_Flow': x1, 'x2_Pc': x2, 'x3_Pp': x3,
                'T_out': T_out, 'T_hi': T_hi, 'T_ho': T_ho,
                'T_ci': T_ci, 'T_co': T_co, 'dT_LMTD': dT,
                'Q_cond_kw': Q_cond_kw, 'Q_evap_kw': Q_evap_kw,
                'A_cond_m2': A_cond, 'A_evap_m2': A_evap,
                'C_hex_USD': C_hex, 'C_turb_USD': C_turb,
                'C_water_USDyr': C_water,
                'P_turb_kW': P_turb, 'P_pump_kW': P_pump,
                'GC_kW': GC, 'earnings_USDyr': earnings,
                'total_expend_USDyr': total_expend,
                'net_rev_USDyr': net_rev,
            }

            results_list.append(result)

        except Exception as e:
            print(f"⚠️ 样本 {i+1} 运行失败：{e}")
            results_list.append(None)
            valid_flags.append(False)
            continue

    return results_list, valid_flags


In [ ]:
def expected_improvement(mu, sigma, best_y, xi=0.05):
    imp = mu - best_y - xi
    Z = imp / sigma
    ei = imp * norm.cdf(Z) + sigma * norm.pdf(Z)
    print(ei.shape)
    return ei


def set_seed(seed=42):
    import random, os
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)

def to_numpy(x):
    return x.detach().cpu().numpy() if isinstance(x, torch.Tensor) else np.asarray(x)

In [ ]:
best_idx = np.argmax(y)
best_x_origin = X[best_idx]
best_y_origin = y[best_idx]
print(best_y_origin)
print(best_x_origin)

best_idx = np.argmax(y_initial)
best_x_initial = x_initial[best_idx]
best_y_initial = y_initial[best_idx]
print(best_y_initial)
print(best_x_initial)


In [ ]:
# AP8.8版替换下面的36.0为34.0; 9.0替换为35.0; 10.0替换为36.0； 11.0替换为37.0
Application = win32.Dispatch('Apwn.Document.38.0')
# 获取当前文件夹的地址，三效蒸发器文件和本程序文件需放置在同一个文件夹
address = os.getcwd()
# AP的bkp文件的文件名，即三效蒸发器bkp文件的文件名
SimulationName = 'ORC1_rundata'
# 打开三效蒸发器文件
Application.InitFromArchive2(os.path.abspath(SimulationName + '.bkp'))
# 设置AP用户界面的可见性，1为可见，0为不可见
Application.Visible = 1
# 压制对话框的弹出，1为压制；0为不压制
Application.SuppressDialogs = 1
# 试运行三效蒸发器模拟
Application.Engine.Run2(1)
# 因为AP运行比较慢，所以用一个循环语句，每两秒钟检查一次是否运行完毕
while Application.Engine.IsRunning == 1:
    time.sleep(2)

In [ ]:
def process_aspen_results(results_batch, valid_flags):
    """
    过滤 Aspen 模拟结果，提取有效的净收益值与对应索引

    参数
    ----------
    results_batch : list[dict or None]
        run_aspen_batch 返回的结果列表，每个元素是单次模拟结果字典或 None
    valid_flags : list[bool]
        run_aspen_batch 返回的收敛状态列表，True 表示收敛

    返回
    ----------
    next_y_orig_batch : np.ndarray
        收敛样本的目标函数值数组（净收益）
    valid_indices : list[int]
        收敛样本在原批次中的索引位置
    """
    if len(results_batch) != len(valid_flags):
        raise ValueError("❌ results_batch 与 valid_flags 长度不一致！")

    next_y_list = []
    valid_indices = []

    for idx, (result, flag) in enumerate(zip(results_batch, valid_flags)):
        if flag and result is not None:
            # 提取目标值
            next_y_list.append(float(result.get("net_rev_USDyr", np.nan)))
            valid_indices.append(idx)

    # 转为 numpy 数组（便于后续计算）
    next_y_orig_batch = np.array(next_y_list, dtype=float)

    if len(next_y_orig_batch) == 0:
        print("⚠️ 本轮无有效 Aspen 模拟结果。")

    return next_y_orig_batch, valid_indices


In [ ]:
# ---- 单次 BO 运行函数：返回每次迭代的 best_y 列表，并且随机初始化训练集 ----
def run_bo_once(seed, N_ITERS=20, BATCH_SIZE_NEXT=5):
    print("开始优化循环")
    set_seed(seed)

    # 将候选集/训练集都转为 numpy （避免混用 torch 与 np）
    X_cand_orig = to_numpy(X_candidate).copy()
    # y_cand_orig = to_numpy(y_candidate).copy()
    X_cand_scaled = to_numpy(X_candidate_normalized).copy()
    # y_cand_scaled = to_numpy(y_candidate_normalized).copy()

    # x_sample = x_initial
    # y_sample = y_initial
    X_train_scaled = to_numpy(x_initial).copy()
    y_train_scaled = to_numpy(y_initial).ravel().copy()

    best_x = to_numpy(best_x_origin).copy()
    best_y = float(best_y_origin)

    best_y_list = [best_y]  # 记录含初始的 best_y
    best_x_list = [best_x]
    results_all = []
    flage_all = []

    # 可选：记录每次 EI 耗时
    ei_times = []

    for it in range(1, N_ITERS + 1):
        print(f"Running trial {it+1}/{N_ITERS + 1}")
        # 1) Train GP (将训练数据转为 torch)
        x_train_t = torch.tensor(X_train_scaled, dtype=torch.float32)
        y_train_t = torch.tensor(y_train_scaled, dtype=torch.float32)

        # 可在 BaseGP 中传入一个种子参数（若 BaseGP 支持）
        nngp = NNGP(x_train_t, y_train_t, lr=0.01, seed=it) #KANGP BaseGP
        nngp.train_model(epochs=1000)

        
        # 2) Predict on candidate set (把候选 scaled 转 torch)
        X_cand_t = torch.tensor(X_cand_scaled, dtype=torch.float32)
        mean_t, std_t = nngp.predict(X_cand_t, return_std=True)
        mean_cand = to_numpy(mean_t).ravel()
        std_cand = to_numpy(std_t).ravel()

        # 3) best_y -> scaled (使用你已有的 scaler_y)
        best_y_scaled = float(scaler_y.transform(np.array([[best_y]]))[0, 0])
        # 4) compute EI & time it
        t0 = time.perf_counter()
        EI = expected_improvement(mean_cand, std_cand, best_y_scaled, xi=0.01)
        t1 = time.perf_counter()
        ei_times.append(t1 - t0)

        # 5) choose top-N by EI (直接取最大前 N)
        # 注意：np.argsort 返回从小到大索引，取最后 N 个即为最大 N 个
        # 我这里倒序，以便先获得最大值索引顺序（方便检查）
        chosen_idxs = np.argsort(EI)[-BATCH_SIZE_NEXT:][::-1]
        print(chosen_idxs)

        # 6) map to original-scale x,y,这里是得到最大的EI下原始的X候选集的数据
        next_x_orig_batch = X_cand_orig[chosen_idxs].copy()
        print(next_x_orig_batch)

        # 批量运行 Aspen 模拟
        results_batch, valid_flags = run_aspen_batch(next_x_orig_batch, Application, address)
        # 过滤结果并提取有效索引与净收益
        next_y_orig_batch, valid_indices = process_aspen_results(results_batch, valid_flags)
        
        # 若本轮有有效数据，则更新训练集
        if len(valid_indices) > 0:
            next_x_valid_batch = next_x_orig_batch[valid_indices]
            X_train_scaled = np.vstack([X_train_scaled, scaler_X.transform(next_x_valid_batch)])
            y_train_scaled = np.concatenate([y_train_scaled, scaler_y.transform(next_y_orig_batch.reshape(-1, 1)).ravel()])
        else:

            print("⚠️ 本轮无有效数据，跳过。")
            best_y_list.append(best_y)
            best_x_list.append(best_x)
            continue

        # 9) update best_y (original scale)
        batch_best_idx = np.argmax(next_y_orig_batch)
        if next_y_orig_batch[batch_best_idx] > best_y:
            best_y = next_y_orig_batch[batch_best_idx]
            best_x = next_x_orig_batch[batch_best_idx].copy()

        best_y_list.append(best_y)
        best_x_list.append(best_x)
        results_all.append(results_batch)
        flage_all.append(valid_flags)      

    return best_y_list, best_x_list, ei_times,results_all, flage_all

In [ ]:
# ---- 主循环：重复 N_RUNS 次并保存结果  不同的初始化优化过程 ----
from datetime import datetime

N_RUNS = 3
N_ITERS = 10
BATCH_SIZE_NEXT = 5

all_runs = []
all_runs_x = []
all_ei_times = []
results_all_total = []
flage_all_total = []

for r in range(N_RUNS):
    seed_r = r  # 可按需设置不同种子
    print(f"Running trial {r+1}/{N_RUNS} with seed {seed_r} ...")
    best_y_list, best_x_list, ei_times,results_all, flage_all = run_bo_once(seed=seed_r, N_ITERS=N_ITERS, BATCH_SIZE_NEXT=BATCH_SIZE_NEXT)
    all_runs.append(best_y_list)        # each is length N_ITERS+1
    all_runs_x.append(best_x_list)
    all_ei_times.append(ei_times)      # each is length N_ITERS
    results_all_total.append(results_all)
    flage_all_total.append(flage_all)


# 转成 numpy 矩阵： shape = (N_RUNS, N_ITERS+1)
y_all = np.array(all_runs)
# y_all = np.atleast_2d(np.array(all_runs))
print("y_all.shape:", y_all.shape)

# 保存原始每-run数据到 CSV（列：iteration, run_1, run_2, ...）
df_results = pd.DataFrame({"iteration": list(range(N_ITERS+1))})
for i in range(N_RUNS):
    df_results[f"run_{i+1}"] = y_all[i, :]

df_results.to_csv("stdkangpBO_multi_runs.csv", index=False)
print("Saved stdkangpBO_multi_runs.csv")

# ---- 计算均值、SEM、95% CI ----
N_TRIALS = N_RUNS
y_mean = y_all.mean(axis=0)         # shape (N_ITERS+1,)
y_sem = y_all.std(axis=0, ddof=1) / np.sqrt(N_TRIALS)
ci95 = 1.96 * y_sem
lower = y_mean - ci95
upper = y_mean + ci95

# 保存均值+CI到 CSV
df_stats = pd.DataFrame({
    "iteration": list(range(N_ITERS+1)),
    "mean": y_mean,
    "sem": y_sem,
    "ci95_lower": lower,
    "ci95_upper": upper
})
df_stats.to_csv("kangpBO_multi_runs_stats.csv", index=False)
print("Saved kangpBO_multi_runs_stats.csv")

# ---- 绘图：均值曲线 + 95% CI 阴影 ----
plt.figure(figsize=(8, 5))
iters = np.arange(N_ITERS + 1)
plt.plot(iters, y_mean, marker='o', label='Mean best_y', color='C0')
plt.fill_between(iters, lower, upper, color='C0', alpha=0.25, label='95% CI')
plt.xlabel('Number of Iterations')
plt.ylabel('Best Objective Value')
plt.title(f'BO Progress (N_runs={N_RUNS}, batch={BATCH_SIZE_NEXT})')
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()
plt.xticks(iters)
plt.savefig('stdbo_mean_ci.png', dpi=300)
plt.show()

# ---- 可选：打印 EI 时间统计（每次迭代的平均 EI 时间） ----
# all_ei_times: list of length N_RUNS, each element length N_ITERS
ei_mat = np.array(all_ei_times)  # shape (N_RUNS, N_ITERS)
ei_mean_per_iter = ei_mat.mean(axis=0)
ei_sem_per_iter = ei_mat.std(axis=0, ddof=1) / np.sqrt(N_TRIALS)

df_eitime = pd.DataFrame({
    "iteration": list(range(1, N_ITERS+1)),
    "ei_time_mean": ei_mean_per_iter,
    "ei_time_sem": ei_sem_per_iter
})
df_eitime.to_csv("kangpBO_ei_time_stats.csv", index=False)
print("Saved kangpBO_ei_time_stats.csv")

#-----记录每一次的结果-----
# 假设你的变量已经存在：results_all_total, flage_all_total
# 注意：不要先用 np.array(...) 覆盖原始 list，否则可能把 dict 强转为 object array 难处理
df_all = flatten_bo_results(results_all_total, flage_all_total, verbose=True)
# 保留三位小数（只针对数值列）
num_cols = df_all.select_dtypes(include=[float, int]).columns
df_all[num_cols] = df_all[num_cols].round(3)

# 保存
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
out_fname = f"nnGP_BO_all_runs_flattened_{timestamp}.xlsx"
df_all.to_excel(out_fname, index=False)
print("保存完成：", out_fname)
print("前 3 行预览：")
print(df_all.head(3))
print("总行数：", df_all.shape[0])



In [ ]:
print(y_all)
print(y_all.shape)

In [ ]:
import psutil

def get_pid(process_name):
   for proc in psutil.process_iter():
       if proc.name() == process_name:
          return proc.pid

process_name = "AspenPlus.exe"
p = psutil.Process(get_pid(process_name))
p.terminate()